# 03. contextlib

**Tier:** Intermediate. **Why you'd use this:** You want a small `with` block that times a section or opens a temporary handle without writing a full class. `@contextmanager` builds that from a generator.

**After this lesson you can:** write a `@contextmanager` generator; put cleanup in `finally` after `yield`; know when a class is clearer.
About 25 minutes. **Uses:** [Context managers](02-context-managers-lesson.ipynb), [Generators](../07-iteration/02-generators-lesson.ipynb)


## Quick review

1. When does `__exit__` run?
2. What does returning `True` from `__exit__` mean?

<details><summary>Answers</summary>

After the `with` body, even on error. Suppress the exception.
</details>


## The idea

`@contextmanager` wraps a generator function. Everything before `yield` is setup. The yielded value becomes the `as` target. Everything after `yield` is cleanup. Use `try`/`finally` around `yield` so cleanup still runs when the body fails.

<details><summary>Explain more</summary>

The decorator builds an object with `__enter__` and `__exit__` that drive your generator. You write the linear story once.
</details>


## See it work

Mark enter and exit with a generator-based manager.


In [1]:
from contextlib import contextmanager


@contextmanager
def marked():
    # 1. Setup before yield, cleanup in finally after.
    events = ["open"]
    try:
        yield events
    finally:
        events.append("close")


with marked() as events:
    events.append("inside")
print(events)


['open', 'inside', 'close']


## Check yourself

Answer these before you run the cell.

1. What does `yield` supply to the `as` target?
2. Why wrap `yield` in `try`/`finally`?


In [2]:
from contextlib import contextmanager


@contextmanager
def marked():
    events = ["open"]
    try:
        yield events
    finally:
        events.append("close")


try:
    with marked() as events:
        events.append("inside")
        raise RuntimeError("boom")
except RuntimeError:
    print(events)


['open', 'inside', 'close']


<details><summary>Answers</summary>

The yielded object. So cleanup still runs when the body raises.
</details>


## Why it works this way

`contextlib.contextmanager` turns a generator into a context manager. Code before `yield` is `__enter__`. The yielded value is the `as` target. Code after `yield` is `__exit__` cleanup. If the body raises, the exception is thrown into the generator at `yield`, so a `try`/`finally` around `yield` still runs cleanup. Deep dive: [The iteration protocol](../../internals/07-iteration/01-protocol-lesson.ipynb).

This form avoids a full class when the resource story is linear. It still depends on the generator protocol: one yield, then cleanup. Forgetting `try`/`finally` around `yield` means cleanup is skipped on error, which defeats the point of `with`.

Cost is a generator object per `with`. The design reason is less boilerplate for simple pairings. Prefer a class when you need reusable state, subclassing, or several public methods on the manager. Prefer `@contextmanager` when the story is setup, yield a value, clean up.

Evidence: a marker list that appends `"close"` after the body, and the same close after a raised error when `finally` guards the cleanup.

Generator-based managers shine for local helpers inside a module: temporary chdir, temporary env vars, capturing logs, timing blocks. They keep the happy path readable as a straight-line story. The decorator hides the class scaffolding, not the need for correct cleanup. If you find yourself storing many attributes on the generator, that is a signal to write a real class instead.

`contextlib` also offers helpers like `closing` and `ExitStack` for harder cases. This lesson sticks to `@contextmanager` because it teaches the mapping to `__enter__` and `__exit__` most clearly. Once you can see that mapping, the rest of the toolkit is easier to learn. Always ask: where is setup, what is yielded, where is cleanup, and does cleanup run on error?

Write tests that raise inside the `with` and assert the cleanup side effect. That test is the difference between a manager that works in demos and one that works in production.


In [3]:
from contextlib import contextmanager


@contextmanager
def marked():
    events = ["open"]
    try:
        yield events
    finally:
        events.append("close")


with marked() as events:
    events.append("inside")
print(events)

try:
    with marked() as events:
        events.append("inside")
        raise RuntimeError("boom")
except RuntimeError:
    print(events)


['open', 'inside', 'close']
['open', 'inside', 'close']


## When to use it

Use `@contextmanager` for a short linear setup/cleanup story. Use a class when the manager needs richer state or an API beyond one `with`. Always clean up in `finally` after `yield`. Don't yield more than once inside the generator. If the generator needs complex branching before yield, a class manager will read more clearly.


## Common mistake

Putting cleanup after `yield` without `finally`, so an error in the body skips the close.


In [4]:
from contextlib import contextmanager


@contextmanager
def marked():
    events = ["open"]
    yield events
    events.append("close")


try:
    with marked() as events:
        events.append("inside")
        raise RuntimeError("boom")
except RuntimeError:
    print(events)


['open', 'inside']


Guard cleanup with `finally` so it runs on success and on error.


In [5]:
from contextlib import contextmanager


@contextmanager
def marked():
    events = ["open"]
    try:
        yield events
    finally:
        events.append("close")


try:
    with marked() as events:
        events.append("inside")
        raise RuntimeError("boom")
except RuntimeError:
    print(events)


['open', 'inside', 'close']


## Try it

Write a `@contextmanager` that yields a temporary dict and clears it in `finally`. What does the caller see after the `with` ends?


## Recap

- `@contextmanager` builds a context manager from a generator.
- Setup before `yield`, cleanup in `finally` after.
- Prefer a class when the manager needs a wider API.

**Next:** [practice](03-contextlib-practice.ipynb)
